In [5]:
import yaml
from PIL import Image
from tqdm import tqdm
from segmentation_dito import setup_model
import tensorflow as tf
import time


from utils import plot_grid, mask_on_image
from utils import get_n_confounded_m_non_confounded_images
from segmentation_dito import segment_instance_dito

## Datasets
### Load ISIC 2019 dataset

In [6]:
# Load the YAML file into a dictionary
dataset_yaml_file = "/workspace/home/data/datasets.yaml"
with open(dataset_yaml_file, "r") as file:
    datasets = yaml.safe_load(file)


# get dataset
dataset_key = "ISIC2019"
dataset = datasets[dataset_key]

## Setup images, custom vocabulary and models

In [ ]:
# get n confounded and m non-confounded images
images_path = dataset["raw_path"]
gt_csv_path = dataset["gt_label_path"]

n = 50
m = 50

imgs_confounded, imgs_non_confounded = get_n_confounded_m_non_confounded_images(images_path, gt_csv_path, n=n, m=m)
images = imgs_confounded + imgs_non_confounded


device = 0

# setup custom vocabulary
custom_vocabulary = [
    "colorful patch",
    "colorful sticker",
    #"colorful cutted sticker",
    #"colorful cutted patch",
    "colorful circle",
    "colorful cutted circle",
]

model, text_embeddings, parser_fn = setup_model(custom_vocabulary, device)
confidence_threshold = 0.0007

100%|██████████| 4/4 [00:46<00:00, 11.57s/it]


## Run model on images and visualize the segmentation

In [ ]:
start_time = time.time()
for i, img_dict in enumerate(tqdm(images)):
    mask = segment_instance_dito(img_dict, 
                                custom_vocabulary, 
                                confidence_threshold,
                                model,
                                parser_fn,
                                text_embeddings)

    # visualize output
    print(f"Image {i}: {img_dict['file_name']}")
    img = Image.open(img_dict['file_path']) 
    mask_on_img = mask_on_image(img, mask)
    plot_grid([img, mask, mask_on_img], label=["Original Image", "Segmentation Mask", "Mask on Image"], ncols=3)
    
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")

100%|██████████| 10/10 [04:58<00:00, 29.81s/it]

Time taken: 298.11473274230957 seconds
